# Notebook 01 — Setup i Pobieranie Danych

**Projekt:** Współzależność rynków aktywów cyfrowych i tradycyjnych systemów finansowych — analiza przy użyciu Ensemble Learning  
**Autor:** Praca Magisterska  
**Data:** 2026-03-05

---

## Cel notebooka

Ten notebook realizuje pierwszą fazę projektu:
1. Instalację wszystkich wymaganych bibliotek
2. Pobranie danych finansowych przez API (BTC, WIG20, VIX, DXY, Fed Funds Rate)
3. Wyrównanie indeksów czasowych i synchronizację kalendarzy giełdowych
4. Podstawowe czyszczenie danych (NaN, duplikaty, outliers)
5. Statystyki opisowe i zapis do plików

## Dane
- **BTC/USD:** 2019-01-01 → 2024-12-31 (dzienny + 1H)
- **WIG20:** 2019-01-01 → 2024-12-31 (dzienny)
- **Zmienne makro:** VIX, DXY, Fed Funds Rate

## Uwaga metodologiczna
BTC handluje 24/7 (365 dni), WIG20 tylko w dni robocze GPW (~252 dni/rok). Strategia synchronizacji: zachowujemy tylko dni, gdy GPW było otwarte (dane WIG20 nie są NaN), a dla BTC stosujemy forward-fill na weekendy i święta.

In [1]:
# ============================================================
# KROK 0: Reprodukowalność — ustawienie seedów
# ============================================================
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Ustaw backend Keras na PyTorch (TensorFlow nie obsługuje Python 3.13)
os.environ['KERAS_BACKEND'] = 'torch'

# Spróbuj ustawić seed PyTorch
try:
    import torch
    torch.manual_seed(SEED)
    print(f"PyTorch seed ustawiony (backend: torch, MPS: {torch.backends.mps.is_available()})")
except ImportError:
    print("PyTorch niedostępny")

print(f"Python random seed: {SEED}")
print(f"NumPy seed: {SEED}")

PyTorch seed ustawiony (backend: torch, MPS: True)
Python random seed: 42
NumPy seed: 42


## 1. Instalacja bibliotek

Instalujemy wszystkie wymagane pakiety z pliku `requirements.txt`. Instalacja jest idempotentna — ponowne uruchomienie nic nie zmieni jeśli biblioteki są już zainstalowane.

In [2]:
# ============================================================
# KROK 1: Instalacja bibliotek
# ============================================================
import subprocess
import sys

print("Instalacja bibliotek z requirements.txt...")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "--quiet"],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print("Instalacja zakonczona pomyslnie.")
else:
    print("Bledy podczas instalacji:")
    print(result.stderr[-2000:])  # ostatnie 2000 znaków błędów

# Weryfikacja kluczowych bibliotek
import importlib
libs_to_check = [
    'pandas', 'numpy', 'scipy', 'statsmodels', 'arch',
    'yfinance', 'ccxt', 'sklearn', 'xgboost', 'lightgbm',
    'matplotlib', 'seaborn', 'plotly', 'joblib', 'tqdm'
]

print("\nWeryfikacja wersji bibliotek:")
print("-" * 50)
for lib in libs_to_check:
    try:
        mod = importlib.import_module(lib)
        version = getattr(mod, '__version__', 'unknown')
        print(f"  {lib:<20} v{version}")
    except ImportError:
        print(f"  {lib:<20} *** NIE ZAINSTALOWANY ***")

Instalacja bibliotek z requirements.txt...


Instalacja zakonczona pomyslnie.

Weryfikacja wersji bibliotek:
--------------------------------------------------


  pandas               v2.2.3
  numpy                v2.1.3
  scipy                v1.15.3


  statsmodels          v0.14.4


  arch                 v8.0.0
  yfinance             v1.2.0


  ccxt                 v4.3.98
  sklearn              v1.6.1


  xgboost              v3.2.0


  lightgbm             v4.6.0
  matplotlib           v3.10.0
  seaborn              v0.13.2
  plotly               v5.24.1
  joblib               v1.4.2
  tqdm                 v4.67.1


In [3]:
# ============================================================
# KROK 1b: Import wszystkich bibliotek
# ============================================================
import os
import time
import warnings
import json
from pathlib import Path
from datetime import datetime, timedelta

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import seaborn as sns
from scipy import stats
import yfinance as yf
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.4f}'.format)

# ── Globalny styl wykresów ──────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')
mpl.rcParams['font.family'] = 'DejaVu Sans'
mpl.rcParams['font.size'] = 11
mpl.rcParams['axes.titlesize'] = 13
mpl.rcParams['axes.labelsize'] = 11
mpl.rcParams['figure.dpi'] = 150

COLOR_BTC = '#F7931A'    # Bitcoin orange
COLOR_WIG = '#003087'    # GPW blue
COLOR_HRI = '#C0392B'    # Risk red
COLOR_VIX = '#27AE60'    # Green
COLOR_DXY = '#8E44AD'    # Purple

# ── Ścieżki do katalogów ────────────────────────────────────
BASE_DIR = Path('.')
DATA_RAW = BASE_DIR / 'data' / 'raw'
DATA_PROC = BASE_DIR / 'data' / 'processed'
RESULTS_TABLES = BASE_DIR / 'results' / 'tables'
RESULTS_FIGS = BASE_DIR / 'results' / 'figures'
MODELS_DIR = BASE_DIR / 'models'

# Utwórz katalogi jeśli nie istnieją
for d in [DATA_RAW, DATA_PROC, RESULTS_TABLES, RESULTS_FIGS, MODELS_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print("Biblioteki zaladowane pomyslnie.")
print(f"Katalog roboczy: {Path('.').resolve()}")
print(f"Pandas:  v{pd.__version__}")
print(f"NumPy:   v{np.__version__}")
print(f"yfinance: v{yf.__version__}")

Biblioteki zaladowane pomyslnie.
Katalog roboczy: /Users/karcyh/Desktop/praca magisterska
Pandas:  v2.2.3
NumPy:   v2.1.3
yfinance: v1.2.0


## 2. Definicja funkcji pomocniczych

Przed pobraniem danych definiujemy dwie kluczowe funkcje:
- `fetch_with_retry()` — pobieranie danych z retry logic (max 3 próby, exponential backoff)
- `save_raw()` — zapis surowych danych do `data/raw/` natychmiast po pobraniu

In [4]:
# ============================================================
# KROK 2: Funkcje pomocnicze — retry logic i zapis
# ============================================================

def fetch_with_retry(fetch_func, *args, max_retries=3, base_delay=2.0, **kwargs):
    """
    Wrapper z retry logic (exponential backoff).
    
    Parameters
    ----------
    fetch_func : callable
        Funkcja pobierająca dane (np. yf.download)
    *args : positional args
        Argumenty przekazywane do fetch_func
    max_retries : int
        Maksymalna liczba prób (domyślnie 3)
    base_delay : float
        Bazowe opóźnienie w sekundach przed retry
    **kwargs : keyword args
        Argumenty kluczowe przekazywane do fetch_func
    
    Returns
    -------
    DataFrame lub None jeśli wszystkie próby się nie powiodły
    """
    for attempt in range(1, max_retries + 1):
        try:
            print(f"  Proba {attempt}/{max_retries}...", end=' ')
            result = fetch_func(*args, **kwargs)
            if result is not None and not (hasattr(result, 'empty') and result.empty):
                print(f"OK ({len(result)} rekordow)")
                return result
            else:
                print("Brak danych")
        except Exception as e:
            print(f"Blad: {e}")
            if attempt < max_retries:
                delay = base_delay * (2 ** (attempt - 1))  # exponential backoff
                print(f"  Oczekiwanie {delay:.1f}s przed ponowna proba...")
                time.sleep(delay)
    
    print(f"  NIEPOWODZENIE po {max_retries} probach.")
    return None


def save_raw(df, filename, description=""):
    """
    Zapisuje surowe dane do data/raw/ jako CSV.
    
    Parameters
    ----------
    df : DataFrame
        Dane do zapisania
    filename : str
        Nazwa pliku (bez rozszerzenia)
    description : str
        Opis zapisywanych danych
    """
    filepath = DATA_RAW / f"{filename}.csv"
    df.to_csv(filepath)
    print(f"  Zapisano: {filepath} ({len(df)} wierszy, {df.shape[1]} kolumn) {description}")
    return filepath


def load_cached_or_fetch(filename, fetch_func, *args, force_reload=False, **kwargs):
    """
    Ładuje dane z cache (CSV) jeśli istnieje, lub pobiera świeże dane.
    Pozwala to na szybkie ponowne uruchomienie notebooka bez ponownego pobierania.
    """
    filepath = DATA_RAW / f"{filename}.csv"
    
    if filepath.exists() and not force_reload:
        print(f"  Ladowanie z cache: {filepath}")
        df = pd.read_csv(filepath, index_col=0, parse_dates=True)
        print(f"  Zaladowano {len(df)} rekordow z cache.")
        return df
    
    print(f"  Pobieranie swiezych danych: {filename}...")
    df = fetch_with_retry(fetch_func, *args, **kwargs)
    if df is not None:
        save_raw(df, filename)
    return df


print("Funkcje pomocnicze zdefiniowane.")

Funkcje pomocnicze zdefiniowane.


## 3. Pobieranie danych

### 3.1 Dane BTC/USD — Yahoo Finance

Pobieramy dzienne ceny OHLCV Bitcoin od 2019-01-01 do 2024-12-31. Ticker w Yahoo Finance: `BTC-USD`.

Strategia fallback:
1. Yahoo Finance (`yfinance`) — primary source
2. Jeśli niedostępne → sprawdzamy czy istnieje lokalny plik `btc.csv` (już mamy taki plik w projekcie!)
3. Jeśli nic nie działa → błąd krytyczny

In [5]:
# ============================================================
# KROK 3.1: Pobieranie danych BTC/USD — dzienny
# ============================================================

START_DATE = '2019-01-01'
END_DATE   = '2026-08-14'

print("=" * 60)
print("Pobieranie BTC/USD (dzienny) z Yahoo Finance")
print(f"Zakres: {START_DATE} → {END_DATE}")
print("=" * 60)

def fetch_btc_daily():
    """Pobiera dzienne dane BTC/USD z Yahoo Finance."""
    btc = yf.download('BTC-USD', start=START_DATE, end=END_DATE,
                      auto_adjust=True, progress=False)
    # Flatten MultiIndex columns jeśli present
    if isinstance(btc.columns, pd.MultiIndex):
        btc.columns = btc.columns.get_level_values(0)
    btc.index.name = 'Date'
    return btc

btc_daily = load_cached_or_fetch('btc_daily_raw', fetch_btc_daily)

# Fallback: użyj lokalnego pliku btc.csv jeśli Yahoo nie działa
if btc_daily is None:
    print("\nFallback: szukam lokalnego pliku btc.csv...")
    local_btc = Path('btc.csv')
    if local_btc.exists():
        print("  Znaleziono btc.csv — konwertuję na dzienny OHLCV...")
        btc_local = pd.read_csv(local_btc)
        # Wykryj format timestampu
        if 'Timestamp' in btc_local.columns:
            btc_local['Date'] = pd.to_datetime(btc_local['Timestamp'], unit='s')
        elif 'Date' in btc_local.columns:
            btc_local['Date'] = pd.to_datetime(btc_local['Date'])
        else:
            btc_local['Date'] = pd.to_datetime(btc_local.iloc[:, 0])
        
        btc_local = btc_local.set_index('Date')
        
        # Jeśli dane godzinowe — resample do dziennych
        if len(btc_local) > 2500:  # więcej niż ~7 lat danych dziennych
            print("  Dane wydają się być wysokiej częstotliwości — resampling do dziennych...")
            btc_daily = pd.DataFrame({
                'Open':   btc_local['Open'].resample('D').first(),
                'High':   btc_local['High'].resample('D').max(),
                'Low':    btc_local['Low'].resample('D').min(),
                'Close':  btc_local['Close'].resample('D').last(),
                'Volume': btc_local['Volume'].resample('D').sum()
            }).dropna()
        else:
            btc_daily = btc_local[['Open', 'High', 'Low', 'Close', 'Volume']]
        
        # Filtruj zakres dat
        btc_daily = btc_daily.loc[START_DATE:END_DATE]
        save_raw(btc_daily, 'btc_daily_raw', '(z lokalnego pliku btc.csv)')
    else:
        raise RuntimeError("BŁĄD KRYTYCZNY: Nie można pobrać danych BTC i brak lokalnego pliku btc.csv")

# Weryfikacja
btc_daily = btc_daily[btc_daily['Close'] > 0]  # usuń zerowe/nieprawidłowe ceny
btc_daily.index = pd.to_datetime(btc_daily.index)
btc_daily = btc_daily.loc[START_DATE:END_DATE]

print(f"\nDane BTC dzienne:")
print(f"  Zakres: {btc_daily.index.min()} → {btc_daily.index.max()}")
print(f"  Liczba obserwacji: {len(btc_daily)}")
print(f"  Kolumny: {list(btc_daily.columns)}")
print(f"  Brakujące wartości: {btc_daily.isnull().sum().sum()}")
print("\nPierwsze 3 wiersze:")
btc_daily.head(3)

Pobieranie BTC/USD (dzienny) z Yahoo Finance
Zakres: 2019-01-01 → 2026-08-14
  Ladowanie z cache: data/raw/btc_daily_raw.csv
  Zaladowano 2783 rekordow z cache.

Dane BTC dzienne:
  Zakres: 2019-01-01 00:00:00 → 2026-08-14 00:00:00
  Liczba obserwacji: 2783
  Kolumny: ['Open', 'High', 'Low', 'Close', 'Volume']
  Brakujące wartości: 0

Pierwsze 3 wiersze:


,Open,High,Low,Close,Volume
Date,,,,,
2019-01-01,3701.2300,3810.1600,3642.0000,3797.1400,88149249.0923
2019-01-02,3796.4500,3882.1400,3750.4500,3858.5600,133876627.2465
2019-01-03,3857.5700,3862.7400,3730.0000,3766.7800,111657372.6953


In [6]:
# ============================================================
# KROK 3.2: Dane godzinowe BTC — POMINIĘTE (spójność metodologiczna)
# ============================================================
# W pierwotnym przebiegu (marzec 2026) pobieranie danych 1H nie powiodło się
# i cechy btc_1h_* pozostały puste — modele wytrenowano na 52 cechach.
# Yahoo Finance udostępnia interwał 1H wyłącznie dla ostatnich ~730 dni,
# więc uzupełnienie ich teraz objęłoby tylko końcówkę próby i wprowadziłoby
# asymetrię train/test. Krok celowo pominięty.
btc_1h = None
print("Dane 1H pominięte świadomie — cechy btc_1h_* będą puste, jak w pierwotnym przebiegu.")


Dane 1H pominięte świadomie — cechy btc_1h_* będą puste, jak w pierwotnym przebiegu.


In [7]:
# ============================================================
# KROK 3.3: Pobieranie danych WIG20 z Yahoo Finance
# ============================================================

print("=" * 60)
print("Pobieranie WIG20 z Yahoo Finance")
print(f"Zakres: {START_DATE} → {END_DATE}")
print("=" * 60)

def fetch_wig20():
    """Pobiera dzienne dane WIG20 z Yahoo Finance."""
    wig = yf.download('WIG20.WA', start=START_DATE, end=END_DATE,
                      auto_adjust=True, progress=False)
    if isinstance(wig.columns, pd.MultiIndex):
        wig.columns = wig.columns.get_level_values(0)
    wig.index.name = 'Date'
    return wig

# Fallback: stooq.pl przez pandas_datareader
def fetch_wig20_stooq():
    """Pobiera WIG20 ze stooq.pl jako fallback."""
    try:
        import pandas_datareader.data as web
        wig = web.DataReader('wig20.pl', 'stooq', start=START_DATE, end=END_DATE)
        wig = wig.sort_index()
        return wig
    except Exception as e:
        print(f"  Stooq fallback blad: {e}")
        return None

wig20_daily = load_cached_or_fetch('wig20_daily_raw', fetch_wig20)

if wig20_daily is None or len(wig20_daily) < 100:
    print("\nFallback: stooq.pl...")
    wig20_daily = fetch_with_retry(fetch_wig20_stooq)
    if wig20_daily is not None:
        save_raw(wig20_daily, 'wig20_daily_raw', '(stooq.pl fallback)')

if wig20_daily is None:
    raise RuntimeError("BŁĄD KRYTYCZNY: Nie można pobrać danych WIG20")

# Czyszczenie
wig20_daily.index = pd.to_datetime(wig20_daily.index)
if wig20_daily.index.tz is not None:
    wig20_daily.index = wig20_daily.index.tz_localize(None)
wig20_daily = wig20_daily.loc[START_DATE:END_DATE]
wig20_daily = wig20_daily[wig20_daily['Close'] > 0]

# Upewnij się że mamy kolumny Volume (może być 0 dla indeksów)
if 'Volume' not in wig20_daily.columns:
    wig20_daily['Volume'] = np.nan
    print("  UWAGA: Brak danych wolumenu dla WIG20 — uzupełniono NaN")

print(f"\nDane WIG20:")
print(f"  Zakres: {wig20_daily.index.min()} → {wig20_daily.index.max()}")
print(f"  Liczba obserwacji (dni handlowych GPW): {len(wig20_daily)}")
print(f"  Kolumny: {list(wig20_daily.columns)}")
print(f"  Brakujące wartości: {wig20_daily.isnull().sum().to_dict()}")
wig20_daily.head(3)

Pobieranie WIG20 z Yahoo Finance
Zakres: 2019-01-01 → 2026-08-14
  Ladowanie z cache: data/raw/wig20_daily_raw.csv
  Zaladowano 1906 rekordow z cache.

Dane WIG20:
  Zakres: 2019-01-02 00:00:00 → 2026-08-14 00:00:00
  Liczba obserwacji (dni handlowych GPW): 1906
  Kolumny: ['Open', 'High', 'Low', 'Close', 'Volume']
  Brakujące wartości: {'Open': 0, 'High': 0, 'Low': 0, 'Close': 0, 'Volume': 0}


,Open,High,Low,Close,Volume
Date,,,,,
2019-01-02,2275.3500,2303.2400,2243.9200,2301.6200,399789584.0000
2019-01-03,2293.4700,2293.4700,2243.0500,2247.2200,588350301.0000
2019-01-04,2257.4000,2289.1900,2257.4000,2284.9500,561894484.0000


In [8]:
# ============================================================
# KROK 3.4: Pobieranie VIX i DXY
# ============================================================

print("=" * 60)
print("Pobieranie zmiennych makroekonomicznych")
print("=" * 60)

# --- VIX (CBOE Volatility Index) ---
print("\n[1/3] VIX (^VIX)")

def fetch_vix():
    vix = yf.download('^VIX', start=START_DATE, end=END_DATE,
                      auto_adjust=True, progress=False)
    if isinstance(vix.columns, pd.MultiIndex):
        vix.columns = vix.columns.get_level_values(0)
    vix.index.name = 'Date'
    return vix[['Close']].rename(columns={'Close': 'vix'})

vix_data = load_cached_or_fetch('vix_raw', fetch_vix)

if vix_data is not None:
    vix_data.index = pd.to_datetime(vix_data.index)
    if vix_data.index.tz is not None:
        vix_data.index = vix_data.index.tz_localize(None)
    print(f"  VIX: {len(vix_data)} rekordow, zakres: {vix_data.index.min().date()} → {vix_data.index.max().date()}")
    print(f"  VIX min: {vix_data['vix'].min():.1f}, max: {vix_data['vix'].max():.1f}")

# --- DXY (US Dollar Index) ---
print("\n[2/3] DXY (DX-Y.NYB lub UUP)")

def fetch_dxy():
    # Próbuj kilka możliwych tickerów dla DXY
    for ticker in ['DX-Y.NYB', 'UUP', 'DX=F']:
        try:
            dxy = yf.download(ticker, start=START_DATE, end=END_DATE,
                              auto_adjust=True, progress=False)
            if isinstance(dxy.columns, pd.MultiIndex):
                dxy.columns = dxy.columns.get_level_values(0)
            if len(dxy) > 100:
                dxy.index.name = 'Date'
                dxy = dxy[['Close']].rename(columns={'Close': 'dxy'})
                print(f"    Udalo sie z tickerem: {ticker}")
                return dxy
        except Exception as e:
            print(f"    Ticker {ticker} blad: {e}")
    return None

dxy_data = load_cached_or_fetch('dxy_raw', fetch_dxy)

if dxy_data is not None:
    dxy_data.index = pd.to_datetime(dxy_data.index)
    if dxy_data.index.tz is not None:
        dxy_data.index = dxy_data.index.tz_localize(None)
    print(f"  DXY: {len(dxy_data)} rekordow, zakres: {dxy_data.index.min().date()} → {dxy_data.index.max().date()}")

# --- Fed Funds Rate (FRED) ---
print("\n[3/3] Fed Funds Rate (FRED API)")

def fetch_fed_rate():
    try:
        from fredapi import Fred
        # Klucz FRED API — zarejestruj się bezpłatnie na fred.stlouisfed.org
        # Lub ustaw zmienną środowiskową FRED_API_KEY
        fred_key = os.environ.get('FRED_API_KEY', None)
        if fred_key is None:
            print("  OSTRZEZENIE: Brak FRED_API_KEY — próba bez klucza (może nie działać)")
        fred = Fred(api_key=fred_key)
        # FEDFUNDS = efektywna stopa fed funds (miesięczna)
        fed_monthly = fred.get_series('FEDFUNDS', observation_start=START_DATE,
                                       observation_end=END_DATE)
        fed_df = pd.DataFrame({'fed_rate': fed_monthly})
        fed_df.index.name = 'Date'
        # Resampling miesięczny → dzienny (forward-fill)
        date_range = pd.date_range(START_DATE, END_DATE, freq='D')
        fed_daily = fed_df.reindex(date_range).ffill()
        fed_daily.index.name = 'Date'
        return fed_daily
    except Exception as e:
        print(f"  FRED API blad: {e}")
        return None

fed_rate_data = load_cached_or_fetch('fed_rate_raw', fetch_fed_rate)

if fed_rate_data is not None:
    fed_rate_data.index = pd.to_datetime(fed_rate_data.index)
    print(f"  Fed Rate: {len(fed_rate_data)} rekordow")
    print(f"  Zakres wartości: {fed_rate_data['fed_rate'].min():.2f}% → {fed_rate_data['fed_rate'].max():.2f}%")
else:
    print("  Tworzę zastępczy szereg Fed Funds Rate z aproksymacją...")
    # Aproksymacja na podstawie publicznie znanych zmian stóp
    date_range = pd.date_range(START_DATE, END_DATE, freq='D')
    fed_approx = pd.Series(index=date_range, dtype=float)
    # Przybliżone daty i wartości Fed Funds Rate (znane historyczne dane)
    fed_schedule = [
        ('2019-01-01', 2.40), ('2019-09-19', 2.00), ('2019-10-31', 1.75),
        ('2019-12-11', 1.75), ('2020-03-04', 1.25), ('2020-03-16', 0.25),
        ('2022-03-17', 0.50), ('2022-05-05', 1.00), ('2022-06-16', 1.75),
        ('2022-07-28', 2.50), ('2022-09-22', 3.25), ('2022-11-03', 4.00),
        ('2022-12-15', 4.50), ('2023-02-02', 4.75), ('2023-03-23', 5.00),
        ('2023-05-04', 5.25), ('2023-07-27', 5.50), ('2024-09-19', 5.00),
        ('2024-11-08', 4.75), ('2024-12-19', 4.50)
    ]
    current_rate = 2.40
    for date, rate in fed_schedule:
        fed_approx[date:] = rate
    fed_approx = fed_approx.ffill().bfill()
    fed_rate_data = pd.DataFrame({'fed_rate': fed_approx})
    fed_rate_data.index.name = 'Date'
    save_raw(fed_rate_data, 'fed_rate_raw', '(aproksymacja)')
    print(f"  Utworzono aproksymację Fed Rate: {len(fed_rate_data)} obserwacji")

print("\nPobieranie zmiennych makro zakonczone.")

Pobieranie zmiennych makroekonomicznych

[1/3] VIX (^VIX)
  Ladowanie z cache: data/raw/vix_raw.csv
  Zaladowano 1913 rekordow z cache.
  VIX: 1913 rekordow, zakres: 2019-01-02 → 2026-08-14
  VIX min: 11.5, max: 82.7

[2/3] DXY (DX-Y.NYB lub UUP)
  Ladowanie z cache: data/raw/dxy_raw.csv
  Zaladowano 1914 rekordow z cache.
  DXY: 1914 rekordow, zakres: 2019-01-02 → 2026-08-14

[3/3] Fed Funds Rate (FRED API)
  Ladowanie z cache: data/raw/fed_rate_raw.csv
  Zaladowano 2783 rekordow z cache.
  Fed Rate: 2783 rekordow
  Zakres wartości: 0.04% → 5.33%

Pobieranie zmiennych makro zakonczone.


## 4. Synchronizacja kalendarzy i merge danych

Kluczowy etap: BTC handluje 365 dni w roku (24/7), GPW tylko ~252 dni (dni robocze bez świąt polskich).

**Strategia:**
1. Używamy kalendarza WIG20 jako referncyjnego (tylko dni gdy GPW jest otwarte)
2. Dla BTC stosujemy forward-fill (ostatnia znana cena) dla weekendów
3. Dane makro (VIX, DXY) forward-fill na dni handlowe GPW
4. Wynikowy dataset: tylko dni gdy GPW było otwarte (ma cenę WIG20)

In [9]:
# ============================================================
# KROK 4: Synchronizacja kalendarzy i merge
# ============================================================

print("=" * 60)
print("Synchronizacja kalendarzy i merge danych")
print("=" * 60)

# ─── 4.1 Przygotowanie kolumn BTC ───────────────────────────
btc = btc_daily.copy()
# Upewnij się, że kolumny mają właściwe nazwy
btc.columns = [c.lower().strip() for c in btc.columns]
# Standaryzuj nazwy kolumn
col_map = {}
for c in btc.columns:
    if 'open' in c.lower(): col_map[c] = 'btc_open'
    elif 'high' in c.lower(): col_map[c] = 'btc_high'
    elif 'low' in c.lower(): col_map[c] = 'btc_low'
    elif 'close' in c.lower() or 'price' in c.lower(): col_map[c] = 'btc_close'
    elif 'vol' in c.lower(): col_map[c] = 'btc_volume'
btc = btc.rename(columns=col_map)
print(f"BTC kolumny po standaryzacji: {list(btc.columns)}")

# ─── 4.2 Przygotowanie kolumn WIG20 ─────────────────────────
wig = wig20_daily.copy()
wig.columns = [c.lower().strip() for c in wig.columns]
col_map_wig = {}
for c in wig.columns:
    if 'open' in c.lower(): col_map_wig[c] = 'wig_open'
    elif 'high' in c.lower(): col_map_wig[c] = 'wig_high'
    elif 'low' in c.lower(): col_map_wig[c] = 'wig_low'
    elif 'close' in c.lower() or 'price' in c.lower(): col_map_wig[c] = 'wig_close'
    elif 'vol' in c.lower(): col_map_wig[c] = 'wig_volume'
wig = wig.rename(columns=col_map_wig)
print(f"WIG20 kolumny po standaryzacji: {list(wig.columns)}")

# ─── 4.3 Stwórz pełny zakres dat i forward-fill BTC ─────────
# Wszystkie dni kalendarzowe od START do END
full_date_range = pd.date_range(start=START_DATE, end=END_DATE, freq='D')

# Reindeksuj BTC na pełny zakres i forward-fill max 3 dni
btc_reindexed = btc.reindex(full_date_range)
btc_reindexed = btc_reindexed.ffill(limit=3)  # max 3 dni forward-fill
btc_reindexed.index.name = 'Date'

print(f"\nBTC po reindeksowaniu: {len(btc_reindexed)} dni")
print(f"  BTC NaN po ffill: {btc_reindexed['btc_close'].isnull().sum()}")

# ─── 4.4 Reindeksuj dane makro na pełny zakres ───────────────
if vix_data is not None:
    vix_reindexed = vix_data.reindex(full_date_range).ffill(limit=5)
    vix_reindexed.index.name = 'Date'
    print(f"VIX po reindeksowaniu: {vix_reindexed['vix'].isnull().sum()} NaN")

if dxy_data is not None:
    dxy_reindexed = dxy_data.reindex(full_date_range).ffill(limit=5)
    dxy_reindexed.index.name = 'Date'
    print(f"DXY po reindeksowaniu: {dxy_reindexed['dxy'].isnull().sum()} NaN")

if fed_rate_data is not None:
    fed_reindexed = fed_rate_data.reindex(full_date_range).ffill(limit=35)
    fed_reindexed.index.name = 'Date'
    print(f"Fed Rate po reindeksowaniu: {fed_reindexed['fed_rate'].isnull().sum()} NaN")

# ─── 4.5 Merge na kalendarzu WIG20 ──────────────────────────
# WIG20 ma tylko dni handlowe GPW — to jest nasz referencyjny kalendarz
master = wig.copy()

# Dodaj BTC (forward-filled weekendy)
master = master.join(btc_reindexed, how='left')

# Dodaj makro
if vix_data is not None:
    master = master.join(vix_reindexed, how='left')
if dxy_data is not None:
    master = master.join(dxy_reindexed, how='left')
if fed_rate_data is not None:
    master = master.join(fed_reindexed, how='left')

master.index.name = 'Date'
master.sort_index(inplace=True)

print(f"\nMaster dataset po merge:")
print(f"  Kształt: {master.shape}")
print(f"  Zakres: {master.index.min().date()} → {master.index.max().date()}")
print(f"  Kolumny: {list(master.columns)}")

Synchronizacja kalendarzy i merge danych
BTC kolumny po standaryzacji: ['btc_open', 'btc_high', 'btc_low', 'btc_close', 'btc_volume']
WIG20 kolumny po standaryzacji: ['wig_open', 'wig_high', 'wig_low', 'wig_close', 'wig_volume']

BTC po reindeksowaniu: 2783 dni
  BTC NaN po ffill: 0
VIX po reindeksowaniu: 1 NaN
DXY po reindeksowaniu: 1 NaN
Fed Rate po reindeksowaniu: 0 NaN

Master dataset po merge:
  Kształt: (1906, 13)
  Zakres: 2019-01-02 → 2026-08-14
  Kolumny: ['wig_open', 'wig_high', 'wig_low', 'wig_close', 'wig_volume', 'btc_open', 'btc_high', 'btc_low', 'btc_close', 'btc_volume', 'vix', 'dxy', 'fed_rate']


## 5. Czyszczenie danych

Procedura czyszczenia:
1. Usunięcie duplikatów indeksu
2. Obsługa NaN: forward-fill max 3 dni, następnie interpolacja liniowa
3. Wykrycie outlierów (>5 sigma) i oznaczenie flagą
4. Usunięcie wierszy gdzie kluczowe dane (BTC lub WIG20) są nadal NaN

In [10]:
# ============================================================
# KROK 5: Czyszczenie danych
# ============================================================

print("=" * 60)
print("Czyszczenie danych")
print("=" * 60)

df = master.copy()

# ─── 5.1 Duplikaty ──────────────────────────────────────────
n_duplicates = df.index.duplicated().sum()
print(f"\n[1] Duplikaty indeksu: {n_duplicates}")
if n_duplicates > 0:
    df = df[~df.index.duplicated(keep='last')]
    print(f"    Usunięto {n_duplicates} duplikatów")

# ─── 5.2 Obsługa NaN ────────────────────────────────────────
print("\n[2] Obsługa brakujących wartości:")
nan_before = df.isnull().sum()
print("Brakujące wartości PRZED czyszczeniem:")
print(nan_before[nan_before > 0].to_string())

# Forward-fill max 3 dni
df = df.ffill(limit=3)

# Interpolacja liniowa dla reszty (poza końcami szeregu)
numeric_cols = df.select_dtypes(include=[np.number]).columns
df[numeric_cols] = df[numeric_cols].interpolate(method='linear', limit_direction='both', limit=5)

nan_after = df.isnull().sum()
print("\nBrakujące wartości PO czyszczeniu:")
remaining_nan = nan_after[nan_after > 0]
if len(remaining_nan) > 0:
    print(remaining_nan.to_string())
else:
    print("  Brak NaN!")

# Usuń wiersze gdzie BTC lub WIG20 Close nadal NaN (krytyczne dane)
rows_before = len(df)
df = df.dropna(subset=['btc_close', 'wig_close'])
rows_after = len(df)
print(f"\n  Usunięto {rows_before - rows_after} wierszy z NaN w btc_close lub wig_close")
print(f"  Pozostało: {rows_after} obserwacji")

# ─── 5.3 Wykrywanie outlierów (>5 sigma) ────────────────────
print("\n[3] Wykrywanie outlierów (>5 sigma):")

# Oblicz log-returns dla detekcji outlierów
btc_returns_temp = np.log(df['btc_close'] / df['btc_close'].shift(1))
wig_returns_temp = np.log(df['wig_close'] / df['wig_close'].shift(1))

# Zdefiniuj próg 5-sigma
def flag_outliers(series, threshold_sigma=5):
    """Zwraca maskę boolowską True dla wartości >threshold_sigma od średniej."""
    mean = series.mean()
    std = series.std()
    return np.abs(series - mean) > threshold_sigma * std

btc_outlier_mask = flag_outliers(btc_returns_temp)
wig_outlier_mask = flag_outliers(wig_returns_temp)

df['flag_btc_outlier'] = 0
df['flag_wig_outlier'] = 0
df.loc[btc_returns_temp[btc_outlier_mask.fillna(False)].index, 'flag_btc_outlier'] = 1
df.loc[wig_returns_temp[wig_outlier_mask.fillna(False)].index, 'flag_wig_outlier'] = 1

n_btc_outliers = df['flag_btc_outlier'].sum()
n_wig_outliers = df['flag_wig_outlier'].sum()
print(f"  BTC outlierów (>5σ): {n_btc_outliers}")
if n_btc_outliers > 0:
    print("  Daty BTC outlierów:")
    btc_out_dates = df[df['flag_btc_outlier']==1].index
    for d in btc_out_dates:
        ret = btc_returns_temp.get(d, np.nan)
        print(f"    {d.date()}: zwrot = {ret:.4f} ({ret/btc_returns_temp.std():.1f}σ)")

print(f"  WIG20 outlierów (>5σ): {n_wig_outliers}")
if n_wig_outliers > 0:
    print("  Daty WIG20 outlierów:")
    wig_out_dates = df[df['flag_wig_outlier']==1].index
    for d in wig_out_dates:
        ret = wig_returns_temp.get(d, np.nan)
        print(f"    {d.date()}: zwrot = {ret:.4f} ({ret/wig_returns_temp.std():.1f}σ)")

print("\n  UWAGA: Outliery są OZNACZONE flagą, NIE usunięte (zgodnie z metodologią).")
print("  Ekstrema cenowe mają ekonomiczne uzasadnienie (COVID, FTX, SVB).")

print(f"\nDataset po czyszczeniu: {df.shape}")

Czyszczenie danych

[1] Duplikaty indeksu: 0

[2] Obsługa brakujących wartości:
Brakujące wartości PRZED czyszczeniem:
Series([], )

Brakujące wartości PO czyszczeniu:
  Brak NaN!

  Usunięto 0 wierszy z NaN w btc_close lub wig_close
  Pozostało: 1906 obserwacji

[3] Wykrywanie outlierów (>5 sigma):
  BTC outlierów (>5σ): 3
  Daty BTC outlierów:
    2019-05-13: zwrot = 0.2008 (5.0σ)
    2020-03-12: zwrot = -0.5026 (-12.6σ)
    2022-06-13: zwrot = -0.2575 (-6.5σ)
  WIG20 outlierów (>5σ): 4
  Daty WIG20 outlierów:
    2020-03-09: zwrot = -0.0819 (-5.6σ)
    2020-03-12: zwrot = -0.1425 (-9.7σ)
    2022-02-24: zwrot = -0.1150 (-7.8σ)
    2022-02-25: zwrot = 0.0810 (5.5σ)

  UWAGA: Outliery są OZNACZONE flagą, NIE usunięte (zgodnie z metodologią).
  Ekstrema cenowe mają ekonomiczne uzasadnienie (COVID, FTX, SVB).

Dataset po czyszczeniu: (1906, 15)


## 6. Statystyki opisowe

Obliczamy pełne statystyki opisowe dla wszystkich szeregów i zapisujemy do CSV i LaTeX.

In [11]:
# ============================================================
# KROK 6: Statystyki opisowe
# ============================================================

print("=" * 60)
print("Statystyki opisowe")
print("=" * 60)

# Oblicz log-returns
df['btc_log_return'] = np.log(df['btc_close'] / df['btc_close'].shift(1))
df['wig_log_return'] = np.log(df['wig_close'] / df['wig_close'].shift(1))

if 'vix' in df.columns:
    df['vix_change'] = df['vix'].pct_change()
if 'dxy' in df.columns:
    df['dxy_change'] = df['dxy'].pct_change()

# Kolumny do analizy
analysis_cols = {
    'btc_close': 'BTC/USD Close',
    'btc_log_return': 'BTC Log-Return',
    'wig_close': 'WIG20 Close',
    'wig_log_return': 'WIG20 Log-Return',
}
if 'vix' in df.columns:
    analysis_cols['vix'] = 'VIX'
if 'dxy' in df.columns:
    analysis_cols['dxy'] = 'DXY'
if 'fed_rate' in df.columns:
    analysis_cols['fed_rate'] = 'Fed Funds Rate (%)'

# Funkcja obliczająca pełne statystyki
def compute_descriptive_stats(df, cols_dict):
    """
    Oblicza pełny zestaw statystyk opisowych.
    
    Returns DataFrame z wierszami = statystyki, kolumny = szeregi
    """
    results = {}
    
    for col, label in cols_dict.items():
        if col not in df.columns:
            continue
        s = df[col].dropna()
        
        results[label] = {
            'N (obserwacje)': len(s),
            'Średnia': s.mean(),
            'Mediana': s.median(),
            'Odchylenie std': s.std(),
            'Minimum': s.min(),
            'Maksimum': s.max(),
            'Skośność': stats.skew(s),
            'Kurtoza (nadmiarowa)': stats.kurtosis(s),
            'Kwantyl 5%': s.quantile(0.05),
            'Kwantyl 25%': s.quantile(0.25),
            'Kwantyl 75%': s.quantile(0.75),
            'Kwantyl 95%': s.quantile(0.95),
        }
    
    return pd.DataFrame(results)

desc_stats = compute_descriptive_stats(df, analysis_cols)

print("\nStatystyki opisowe:")
print(desc_stats.round(4).to_string())

# Zapis do CSV
csv_path = RESULTS_TABLES / '01_descriptive_stats.csv'
desc_stats.to_csv(csv_path, float_format='%.6f')
print(f"\nZapisano CSV: {csv_path}")

# Zapis do LaTeX
tex_path = RESULTS_TABLES / '01_descriptive_stats.tex'
desc_stats.to_latex(
    str(tex_path),
    float_format='%.4f',
    caption='Statystyki opisowe szeregów czasowych (2019--2024)',
    label='tab:descriptive_stats',
    escape=False,
    index=True,
    bold_rows=False
)
print(f"Zapisano LaTeX: {tex_path}")

Statystyki opisowe

Statystyki opisowe:
                      BTC/USD Close  BTC Log-Return  WIG20 Close  WIG20 Log-Return       VIX       DXY  Fed Funds Rate (%)
N (obserwacje)            1906.0000       1905.0000    1906.0000         1905.0000 1906.0000 1906.0000           1906.0000
Średnia                  43879.8834          0.0015    2279.8573            0.0003   20.0759   99.6929              2.7452
Mediana                  37306.6200          0.0004    2217.5100            0.0001   18.1600   99.1300              3.0800
Odchylenie std           32126.8048          0.0399     520.7080            0.0147    7.3709    4.9698              2.0032
Minimum                   3398.4000         -0.5026    1305.7300           -0.1425   11.5400   89.4400              0.0400
Maksimum                124658.5400          0.2008    4065.1200            0.0810   82.6900  114.1100              5.3300
Skośność                     0.6531         -1.0505       1.0759           -0.7084    2.8239    0.1

In [12]:
# ============================================================
# KROK 6b: Podgląd statystyk rocznych
# ============================================================

print("=" * 60)
print("Roczne statystyki BTC i WIG20")
print("=" * 60)

# Dodaj kolumnę roku
df['year'] = df.index.year

# Roczne zwroty i zmienność
yearly_stats = []

for year in sorted(df['year'].unique()):
    df_yr = df[df['year'] == year]
    
    # BTC
    btc_ret = df_yr['btc_log_return'].dropna()
    btc_ann_ret = (df_yr['btc_close'].iloc[-1] / df_yr['btc_close'].iloc[0] - 1) * 100
    btc_ann_vol = btc_ret.std() * np.sqrt(252) * 100
    
    # WIG20
    wig_ret = df_yr['wig_log_return'].dropna()
    wig_ann_ret = (df_yr['wig_close'].iloc[-1] / df_yr['wig_close'].iloc[0] - 1) * 100
    wig_ann_vol = wig_ret.std() * np.sqrt(252) * 100
    
    yearly_stats.append({
        'Rok': year,
        'BTC zwrot (%)': round(btc_ann_ret, 1),
        'BTC vol ann (%)': round(btc_ann_vol, 1),
        'WIG20 zwrot (%)': round(wig_ann_ret, 1),
        'WIG20 vol ann (%)': round(wig_ann_vol, 1),
        'N obserwacji': len(df_yr)
    })

yearly_df = pd.DataFrame(yearly_stats).set_index('Rok')
print(yearly_df.to_string())

# Zapis rocznych statystyk
yearly_path = RESULTS_TABLES / '01_yearly_stats.csv'
yearly_df.to_csv(yearly_path)
print(f"\nZapisano: {yearly_path}")

Roczne statystyki BTC i WIG20


      BTC zwrot (%)  BTC vol ann (%)  WIG20 zwrot (%)  WIG20 vol ann (%)  N obserwacji
Rok                                                                                   
2019        87.8000          69.1000          -6.6000            15.1000           248
2020       314.5000          81.9000          -9.8000            33.1000           252
2021        47.3000          80.5000          12.9000            17.7000           251
2022       -64.2000          65.9000         -21.6000            31.2000           251
2023       152.3000          46.0000          30.8000            20.7000           250
2024       106.4000          55.6000          -4.8000            19.6000           249
2025        -8.8000          40.4000          43.0000            22.3000           249
2026       -29.9000          44.8000          23.6000            19.8000           156

Zapisano: results/tables/01_yearly_stats.csv


## 7. Wizualizacja podglądu danych

Szybki wykres podglądowy — pełna analiza EDA w notebooku 02.

In [13]:
# ============================================================
# KROK 7: Wykres podglądowy danych
# ============================================================

fig, axes = plt.subplots(3, 2, figsize=(16, 12))
fig.suptitle('Podgląd pobranych danych — 2019-2024', fontsize=15, fontweight='bold', y=0.98)

# 1. BTC cena
ax = axes[0, 0]
ax.plot(df.index, df['btc_close'], color=COLOR_BTC, linewidth=0.8)
ax.set_title('BTC/USD — Cena dzienna')
ax.set_ylabel('USD')
ax.set_yscale('log')
ax.yaxis.set_major_formatter(mpl.ticker.FuncFormatter(lambda x, p: f'${x:,.0f}'))

# 2. WIG20 cena
ax = axes[0, 1]
ax.plot(df.index, df['wig_close'], color=COLOR_WIG, linewidth=0.8)
ax.set_title('WIG20 — Cena dzienna')
ax.set_ylabel('Punkty')

# 3. BTC log-returns
ax = axes[1, 0]
ax.bar(df.index, df['btc_log_return'], color=COLOR_BTC, alpha=0.5, width=1)
ax.set_title('BTC — Log-zwroty dzienne')
ax.set_ylabel('Log-zwrot')
ax.axhline(0, color='black', linewidth=0.5)

# 4. WIG20 log-returns
ax = axes[1, 1]
ax.bar(df.index, df['wig_log_return'], color=COLOR_WIG, alpha=0.5, width=1)
ax.set_title('WIG20 — Log-zwroty dzienne')
ax.set_ylabel('Log-zwrot')
ax.axhline(0, color='black', linewidth=0.5)

# 5. VIX
ax = axes[2, 0]
if 'vix' in df.columns:
    ax.plot(df.index, df['vix'], color=COLOR_VIX, linewidth=0.8)
    ax.axhline(30, color='orange', linestyle='--', linewidth=0.8, label='VIX=30 (elevated)')
    ax.axhline(40, color='red', linestyle='--', linewidth=0.8, label='VIX=40 (extreme)')
    ax.set_title('VIX — Indeks zmienności S&P500')
    ax.set_ylabel('VIX')
    ax.legend(fontsize=8)
else:
    ax.text(0.5, 0.5, 'VIX niedostępny', ha='center', va='center', transform=ax.transAxes)

# 6. Fed Funds Rate
ax = axes[2, 1]
if 'fed_rate' in df.columns:
    ax.plot(df.index, df['fed_rate'], color=COLOR_HRI, linewidth=1.5)
    ax.set_title('Fed Funds Rate (%)')
    ax.set_ylabel('%')
    ax.fill_between(df.index, df['fed_rate'], alpha=0.15, color=COLOR_HRI)
elif 'dxy' in df.columns:
    ax.plot(df.index, df['dxy'], color=COLOR_DXY, linewidth=0.8)
    ax.set_title('DXY — Indeks dolara')
    ax.set_ylabel('DXY')
else:
    ax.text(0.5, 0.5, 'Dane makro niedostępne', ha='center', va='center', transform=ax.transAxes)

# Oznacz zdarzenia kryzysowe na BTC plot
crisis_events = {
    '2020-03-12': 'COVID',
    '2022-05-09': 'Terra/LUNA',
    '2022-11-08': 'FTX',
    '2023-03-10': 'SVB'
}
for date_str, label in crisis_events.items():
    event_date = pd.Timestamp(date_str)
    if df.index.min() <= event_date <= df.index.max():
        axes[0, 0].axvline(event_date, color='red', alpha=0.5, linewidth=0.8)
        axes[0, 0].text(event_date, axes[0, 0].get_ylim()[1]*0.9, 
                        label, rotation=90, fontsize=7, color='red', alpha=0.7)

for ax in axes.flat:
    ax.xaxis.set_major_formatter(mpl.dates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mpl.dates.YearLocator())
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)

plt.tight_layout()
fig_path = RESULTS_FIGS / '00_data_overview.png'
plt.savefig(fig_path, dpi=150, bbox_inches='tight')
plt.show()
print(f"Zapisano wykres: {fig_path}")

Zapisano wykres: results/figures/00_data_overview.png


## 8. Agregacja danych 1H BTC

Z danych godzinowych obliczamy dzienne statystyki wymagane do feature engineering:
- Dziennie: mean, std, min, max log-zwrotów godzinowych
- Te cechy wzbogacają dataset o informacje o śróddziennej zmienności

In [14]:
# ============================================================
# KROK 8: Agregacja danych BTC 1H → dzienne statystyki
# ============================================================

if btc_1h is not None:
    print("Agregacja danych 1H BTC do dziennych statystyk...")
    
    # Upewnij się że indeks jest timezone-naive
    if btc_1h.index.tz is not None:
        btc_1h.index = btc_1h.index.tz_localize(None)
    
    # Oblicz log-zwroty godzinowe
    close_col = 'Close' if 'Close' in btc_1h.columns else 'btc_close'
    if close_col not in btc_1h.columns:
        # Spróbuj znaleźć kolumnę Close
        close_cols = [c for c in btc_1h.columns if 'close' in c.lower()]
        if close_cols:
            close_col = close_cols[0]
        else:
            close_col = btc_1h.columns[0]  # fallback na pierwszą kolumnę
    
    btc_1h['log_return_1h'] = np.log(btc_1h[close_col] / btc_1h[close_col].shift(1))
    
    # Agreguj do dziennych statystyk
    btc_1h_daily = btc_1h['log_return_1h'].resample('D').agg([
        ('btc_1h_return_mean', 'mean'),
        ('btc_1h_return_std', 'std'),
        ('btc_1h_return_min', 'min'),
        ('btc_1h_return_max', 'max'),
        ('btc_1h_n_obs', 'count')  # liczba obserwacji (diagnostyka)
    ])
    
    # Dołącz do master dataset
    df = df.join(btc_1h_daily, how='left')
    
    n_matched = df['btc_1h_return_mean'].notna().sum()
    print(f"  Dopasowano {n_matched} dni z danych 1H")
    print(f"  Zakres 1H: {btc_1h.index.min().date()} → {btc_1h.index.max().date()}")
    print(f"  Dla dni bez danych 1H (pre-2022): kolumny będą NaN")
else:
    print("Dane 1H niedostępne — tworzę puste kolumny.")
    df['btc_1h_return_mean'] = np.nan
    df['btc_1h_return_std'] = np.nan
    df['btc_1h_return_min'] = np.nan
    df['btc_1h_return_max'] = np.nan
    df['btc_1h_n_obs'] = np.nan

print(f"\nKształt datasetu po agregacji 1H: {df.shape}")

Dane 1H niedostępne — tworzę puste kolumny.

Kształt datasetu po agregacji 1H: (1906, 25)


## 9. Zapis finalnego datasetu

Zapisujemy przetworzony master dataset jako `data/processed/master_dataset.csv`.

In [15]:
# ============================================================
# KROK 9: Zapis finalnego datasetu
# ============================================================

print("=" * 60)
print("Zapis finalnego datasetu")
print("=" * 60)

# Usuń pomocniczą kolumnę roku
df_save = df.drop(columns=['year'], errors='ignore')

# Zapisz do CSV
master_path = DATA_PROC / 'master_dataset.csv'
df_save.to_csv(master_path, float_format='%.8f')
print(f"Zapisano master dataset: {master_path}")
print(f"  Wymiary: {df_save.shape}")
print(f"  Zakres: {df_save.index.min().date()} → {df_save.index.max().date()}")
print(f"  Kolumny: {list(df_save.columns)}")

# Podsumowanie brakujących danych w finalnym datasecie
print("\nPodsumowanie NaN w finalnym datasecie:")
nan_summary = df_save.isnull().sum()
nan_pct = (nan_summary / len(df_save) * 100).round(2)
nan_report = pd.DataFrame({
    'N brakujących': nan_summary,
    '% brakujących': nan_pct
})
nan_report = nan_report[nan_report['N brakujących'] > 0]
if len(nan_report) > 0:
    print(nan_report.to_string())
else:
    print("  Brak NaN w kluczowych kolumnach!")

# Zapis metadanych do JSON
metadata = {
    'created': pd.Timestamp.now().isoformat(),
    'start_date': str(df_save.index.min().date()),
    'end_date': str(df_save.index.max().date()),
    'n_observations': len(df_save),
    'n_columns': len(df_save.columns),
    'columns': list(df_save.columns),
    'btc_price_range': [float(df_save['btc_close'].min()), float(df_save['btc_close'].max())],
    'wig20_price_range': [float(df_save['wig_close'].min()), float(df_save['wig_close'].max())],
    'n_btc_outliers': int(df_save.get('flag_btc_outlier', pd.Series([0])).sum()),
    'n_wig_outliers': int(df_save.get('flag_wig_outlier', pd.Series([0])).sum())
}

meta_path = DATA_PROC / 'master_dataset_metadata.json'
with open(meta_path, 'w', encoding='utf-8') as f:
    json.dump(metadata, f, indent=2, ensure_ascii=False)
print(f"\nZapisano metadane: {meta_path}")

# Podgląd finalnego datasetu
print("\nPierwsze 5 wierszy finalnego datasetu:")
df_save.head()

Zapis finalnego datasetu
Zapisano master dataset: data/processed/master_dataset.csv
  Wymiary: (1906, 24)
  Zakres: 2019-01-02 → 2026-08-14
  Kolumny: ['wig_open', 'wig_high', 'wig_low', 'wig_close', 'wig_volume', 'btc_open', 'btc_high', 'btc_low', 'btc_close', 'btc_volume', 'vix', 'dxy', 'fed_rate', 'flag_btc_outlier', 'flag_wig_outlier', 'btc_log_return', 'wig_log_return', 'vix_change', 'dxy_change', 'btc_1h_return_mean', 'btc_1h_return_std', 'btc_1h_return_min', 'btc_1h_return_max', 'btc_1h_n_obs']

Podsumowanie NaN w finalnym datasecie:
                    N brakujących  % brakujących
btc_log_return                  1         0.0500
wig_log_return                  1         0.0500
vix_change                      1         0.0500
dxy_change                      1         0.0500
btc_1h_return_mean           1906       100.0000
btc_1h_return_std            1906       100.0000
btc_1h_return_min            1906       100.0000
btc_1h_return_max            1906       100.0000
btc_1h_n_obs

,wig_open,wig_high,wig_low,wig_close,wig_volume,btc_open,btc_high,btc_low,btc_close,btc_volume,vix,dxy,fed_rate,flag_btc_outlier,flag_wig_outlier,btc_log_return,wig_log_return,vix_change,dxy_change,btc_1h_return_mean,btc_1h_return_std,btc_1h_return_min,btc_1h_return_max,btc_1h_n_obs
Date,,,,,,,,,,,,,,,,,,,,,,,,
2019-01-02,2275.3500,2303.2400,2243.9200,2301.6200,399789584.0000,3796.4500,3882.1400,3750.4500,3858.5600,133876627.2465,23.2200,96.8200,2.4000,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2019-01-03,2293.4700,2293.4700,2243.0500,2247.2200,588350301.0000,3857.5700,3862.7400,3730.0000,3766.7800,111657372.6953,25.4500,96.3100,2.4000,0,0,-0.0241,-0.0239,0.0960,-0.0053,NaN,NaN,NaN,NaN,NaN
2019-01-04,2257.4000,2289.1900,2257.4000,2284.9500,561894484.0000,3767.2000,3823.6400,3703.5700,3792.0100,111034550.6407,21.3800,96.1900,2.4000,0,0,0.0067,0.0167,-0.1599,-0.0012,NaN,NaN,NaN,NaN,NaN
2019-01-07,2292.7600,2331.4500,2292.7600,2331.4500,696193517.0000,3987.6200,4017.9000,3921.5300,3975.4500,126830380.4929,21.4000,95.7100,2.4000,0,0,0.0472,0.0201,0.0009,-0.0050,NaN,NaN,NaN,NaN,NaN
2019-01-08,2330.4200,2330.4200,2303.0500,2325.1900,765173347.0000,3976.7600,4069.8000,3903.0000,3955.1300,154778846.5424,20.4700,95.9000,2.4000,0,0,-0.0051,-0.0027,-0.0435,0.0020,NaN,NaN,NaN,NaN,NaN


In [16]:
# ============================================================
# PODSUMOWANIE NOTEBOOKA 01
# ============================================================

print("=" * 60)
print(f"Notebook 01 zakończony: {pd.Timestamp.now().strftime('%Y-%m-%d %H:%M:%S')}")
print("=" * 60)
print("\nZapisane pliki:")

saved_files = [
    (DATA_RAW / 'btc_daily_raw.csv', 'BTC/USD dane surowe dzienne'),
    (DATA_RAW / 'btc_1h_raw.csv', 'BTC/USD dane surowe 1H'),
    (DATA_RAW / 'wig20_daily_raw.csv', 'WIG20 dane surowe dzienne'),
    (DATA_RAW / 'vix_raw.csv', 'VIX dane surowe'),
    (DATA_RAW / 'dxy_raw.csv', 'DXY dane surowe'),
    (DATA_RAW / 'fed_rate_raw.csv', 'Fed Funds Rate dane'),
    (DATA_PROC / 'master_dataset.csv', 'Master dataset (przetworzony)'),
    (DATA_PROC / 'master_dataset_metadata.json', 'Metadane master datasetu'),
    (RESULTS_TABLES / '01_descriptive_stats.csv', 'Statystyki opisowe CSV'),
    (RESULTS_TABLES / '01_descriptive_stats.tex', 'Statystyki opisowe LaTeX'),
    (RESULTS_TABLES / '01_yearly_stats.csv', 'Roczne statystyki'),
    (RESULTS_FIGS / '00_data_overview.png', 'Wykres podglądu danych'),
]

for filepath, desc in saved_files:
    exists = filepath.exists()
    status = 'OK' if exists else 'BRAK'
    size = f"{filepath.stat().st_size / 1024:.1f} KB" if exists else '---'
    print(f"  [{status}] {filepath.name:<45} ({size}) — {desc}")

print("\n" + "=" * 60)
print("GOTOWE! Przejdź do Notebook 02 — EDA.")
print("=" * 60)

Notebook 01 zakończony: 2026-08-18 20:29:29

Zapisane pliki:
  [OK] btc_daily_raw.csv                             (173.4 KB) — BTC/USD dane surowe dzienne
  [BRAK] btc_1h_raw.csv                                (---) — BTC/USD dane surowe 1H
  [OK] wig20_daily_raw.csv                           (102.5 KB) — WIG20 dane surowe dzienne
  [OK] vix_raw.csv                                   (54.4 KB) — VIX dane surowe
  [OK] dxy_raw.csv                                   (53.8 KB) — DXY dane surowe
  [OK] fed_rate_raw.csv                              (43.3 KB) — Fed Funds Rate dane
  [OK] master_dataset.csv                            (477.1 KB) — Master dataset (przetworzony)
  [OK] master_dataset_metadata.json                  (0.8 KB) — Metadane master datasetu
  [OK] 01_descriptive_stats.csv                      (1.1 KB) — Statystyki opisowe CSV
  [OK] 01_descriptive_stats.tex                      (1.3 KB) — Statystyki opisowe LaTeX
  [OK] 01_yearly_stats.csv                           (0.3 K